# 03 — Categorical Data Analysis: Contingency Tables

Statistical analysis of label–feature and label–label associations.

**Coverage:**
1. Two-way tables: feature × label (chi-square, Cramér's V)
2. Odds ratios with 95% CIs for key 2×2 tables
3. Mantel–Haenszel: stratified association controlling for confounders
4. Label co-occurrence: pairwise chi-square across all label pairs
5. Simpson's paradox check: crude vs. stratified ORs

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, fisher_exact
import sys
sys.path.insert(0, '..')

from src.labels import LABEL_COLS

In [ ]:
df = pd.read_parquet('../data/processed/train_labeled.parquet')
print(df.shape)

In [ ]:
def chi_square_association(df, var1, var2):
    table = pd.crosstab(df[var1], df[var2])
    chi2, p, dof, expected = chi2_contingency(table.values)
    n = table.values.sum()
    cramers_v = np.sqrt(chi2 / (n * (min(table.shape) - 1)))
    return {'chi2': chi2, 'p_value': p, 'dof': dof,
            'cramers_v': cramers_v, 'reject_independence': p < 0.05}

def odds_ratio_2x2(a, b, c, d):
    # Haldane-Anscombe correction if any cell is zero
    if 0 in (a, b, c, d):
        a, b, c, d = a+.5, b+.5, c+.5, d+.5
    OR = (a * d) / (b * c)
    se_log = np.sqrt(1/a + 1/b + 1/c + 1/d)
    lo, hi = np.exp(np.log(OR) - 1.96*se_log), np.exp(np.log(OR) + 1.96*se_log)
    return {'OR': OR, 'CI_lo': lo, 'CI_hi': hi}

In [ ]:
# 1. Feature × label associations
features = ['category', 'state', 'gender']
results = []
for feat in features:
    for label in LABEL_COLS:
        res = chi_square_association(df, feat, label)
        res.update({'feature': feat, 'label': label})
        results.append(res)

assoc_df = pd.DataFrame(results)[['feature','label','cramers_v','p_value','reject_independence']]
assoc_df = assoc_df.sort_values('cramers_v', ascending=False)
print(assoc_df.to_string(index=False))

In [ ]:
# 2. Pairwise label co-occurrence chi-square
from itertools import combinations
cooc_results = []
for la, lb in combinations(LABEL_COLS, 2):
    res = chi_square_association(df, la, lb)
    cooc_results.append({'L_i': la, 'L_j': lb, 'cramers_v': res['cramers_v'],
                         'p_value': res['p_value']})

cooc_df = pd.DataFrame(cooc_results).sort_values('cramers_v', ascending=False)
print(cooc_df.to_string(index=False))

In [ ]:
# 3. Mantel-Haenszel: L_V x L_R stratified by category
# Test whether velocity-burst and ring co-occur after controlling for merchant category
strata = df['category'].unique()
mh_num, mh_den = 0.0, 0.0
for cat in strata:
    s = df[df['category'] == cat]
    table = pd.crosstab(s['L_V'], s['L_R'])
    if table.shape == (2, 2):
        a,b,c,d = table.iloc[0,0], table.iloc[0,1], table.iloc[1,0], table.iloc[1,1]
        n = a+b+c+d
        mh_num += a*d/n
        mh_den += b*c/n

OR_mh = mh_num / mh_den if mh_den else np.nan
print(f'MH common OR (L_V x L_R | category): {OR_mh:.4f}')

# Compare to crude OR
table_crude = pd.crosstab(df['L_V'], df['L_R'])
a,b,c,d = table_crude.values.ravel()
crude = odds_ratio_2x2(a,b,c,d)
print(f'Crude OR: {crude["OR"]:.4f}  95% CI [{crude["CI_lo"]:.4f}, {crude["CI_hi"]:.4f}]')

In [ ]:
# 4. Cramér's V heatmap across label pairs
v_mat = pd.DataFrame(np.eye(len(LABEL_COLS)), index=LABEL_COLS, columns=LABEL_COLS)
for la, lb in combinations(LABEL_COLS, 2):
    v = chi_square_association(df, la, lb)['cramers_v']
    v_mat.loc[la, lb] = v
    v_mat.loc[lb, la] = v

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(v_mat, annot=True, fmt='.3f', vmin=0, vmax=1, cmap='Blues', ax=ax)
ax.set_title("Cramér's V between fraud labels")
plt.tight_layout()